# 03 · Clasificación e interpretabilidad — EH2025

Este notebook **audita los resultados ya obtenidos** por `src/train.py` y `src/explain.py`. No vuelve a entrenar los modelos ni modifica el conjunto de prueba. Para una reproducción completa del experimento, ejecutar los scripts en `src/` y registrar las versiones de dependencias.


In [ ]:
from pathlib import Path

def project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src" / "features.py").exists() and (candidate / "data" / "processed").exists():
            return candidate
    raise FileNotFoundError("Abre VS Code en la raíz del proyecto EH2025 o ejecuta el notebook desde esa carpeta.")

ROOT = project_root()
FIG = ROOT / "outputs" / "figures"
MODELS = ROOT / "outputs" / "models"
print("Proyecto:", ROOT)

In [ ]:
import json
import pandas as pd
import plotly.express as px

rutas = {
    "cv": MODELS / "validacion_cruzada.csv",
    "resumen_cv": MODELS / "resumen_validacion.csv",
    "prueba": MODELS / "metricas_prueba.csv",
    "importancias": MODELS / "importancia_variables.csv",
    "experimento": MODELS / "experimento.json",
}
for nombre, ruta in rutas.items():
    assert ruta.exists(), f"Falta {ruta}; ejecuta python src/train.py y python src/explain.py"
resultados = {k: pd.read_csv(v) for k, v in rutas.items() if k != "experimento"}
experimento = json.loads(rutas["experimento"].read_text(encoding="utf-8"))
print(json.dumps(experimento, indent=2, ensure_ascii=False))

## 1. Diseño del experimento

- Variable objetivo: `inasistencia_escolar` (1: no matriculado / no asiste; 0: asiste).
- Ocho características demográficas, territoriales y del hogar.
- **Separación por `folio`** entre entrenamiento y prueba.
- Cuatro folds de `StratifiedGroupKFold` en entrenamiento.
- Umbral de clasificación fijado inicialmente en 0,50.
- Selección del algoritmo por **Average Precision media de validación**; el test no se utilizó para elegir el ganador.
- La evaluación predictiva no aplica los factores de expansión del diseño muestral.


In [ ]:
print("Selección:", experimento["modelo_seleccionado"])
print("Train:", experimento["train_rows"], "positivos:", experimento["train_positive"])
print("Test:", experimento["test_rows"], "positivos:", experimento["test_positive"])
assert experimento["train_positive"] + experimento["test_positive"] == 208

## 2. Validación cruzada y variación entre folds

In [ ]:
display(resultados["resumen_cv"])
cv = resultados["cv"]
fig = px.box(cv, x="modelo", y="average_precision", points="all",
             title="Variabilidad de Average Precision en cuatro folds",
             labels={"average_precision":"Average Precision","modelo":"Modelo"})
fig.show()

## 3. Evaluación independiente en prueba

In [ ]:
test = resultados["prueba"]
display(test)
comparacion = test.melt(id_vars="modelo", value_vars=["roc_auc", "average_precision", "f1", "precision", "recall"],
                        var_name="metrica",value_name="valor")
px.bar(comparacion, x="metrica", y="valor", color="modelo", barmode="group",
       title="Comparación de clasificadores — prueba independiente").show()
print("Prevalencia de la prueba:", round(experimento["test_prevalence"], 4))

**Lectura:** Random Forest fue seleccionado por AP en validación. En prueba obtuvo AUC-ROC ≈ 0,7815 y AP ≈ 0,0976, pero Precision ≈ 0,1027; estos valores **no justifican** decisiones individuales automatizadas. Para examinar las curvas ROC, PR y matrices de confusión abrir las figuras `06_...html` a `08_...html` dentro de `outputs/figures/`.

## 4. Explainability: importancia por permutación

In [ ]:
imp = resultados["importancias"].sort_values("importancia_media")
display(imp.sort_values("importancia_media", ascending=False))
fig = px.bar(imp, x="importancia_media", y="descripcion", orientation="h",
             error_x="desviacion", title="Permutation Importance respecto a Average Precision",
             labels={"importancia_media":"Disminución de AP","descripcion":"Característica"})
fig.add_vline(x=0, line_dash="dash")
fig.show()

**Interpretación:** edad, área, departamento y tipo de hogar aportan información predictiva en este experimento. Importancias negativas pueden reflejar redundancia, ruido o limitaciones del test; no significan un efecto protector. La importancia no establece causalidad y el análisis de test no se utilizó para reentrenar.

## 5. Reproducibilidad y límites

El pipeline original se reproduce mediante:

```powershell
python src/features.py
python src/train.py
python src/explain.py
```

`src/train.py` requiere LightGBM y scikit-learn. Debido a la escasez de positivos, la incertidumbre de las métricas puede ser elevada. Los modelos `.joblib` se deben cargar solo de fuentes confiables y no es necesario subirlos al repositorio público.